In [12]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../src")   # the notebook is in notebooks/, the code is in src/

import pandas as pd
from load_data import FitbitData, PERSON_ID
import behavior_insights as bi

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load

In [13]:
data = FitbitData().load().clean()
df = data.for_user(PERSON_ID)
print(PERSON_ID, df.shape, df["Date"].min().date(), "to", df["Date"].max().date())
df.head()

1503960366 (30, 10) 2016-04-12 to 2016-05-11


,Id,Date,steps,calories,active_minutes,sleep_hours,sedentary_minutes,light_minutes,fairly_minutes,very_minutes
0,1503960366,2016-04-12,13162.0,1985.0,38.0,5.450000,728.0,328.0,13.0,25.0
1,1503960366,2016-04-13,10735.0,1797.0,40.0,6.400000,776.0,217.0,19.0,21.0
2,1503960366,2016-04-14,10460.0,1776.0,41.0,NaN,1218.0,181.0,11.0,30.0
3,1503960366,2016-04-15,9762.0,1745.0,63.0,6.866667,726.0,209.0,34.0,29.0
4,1503960366,2016-04-16,12669.0,1863.0,46.0,5.666667,773.0,221.0,10.0,36.0


### Quick check

should only be missing 5 days of sleep

In [14]:
print(df.isna().sum())
df.describe().round(1)

Id                   0
Date                 0
steps                0
calories             0
active_minutes       0
sleep_hours          5
sedentary_minutes    0
light_minutes        0
fairly_minutes       0
very_minutes         0
dtype: int64


,Id,Date,steps,calories,active_minutes,sleep_hours,sedentary_minutes,light_minutes,fairly_minutes,very_minutes
count,3.000000e+01,30,30.0,30.0,30.0,25.0,30.0,30.0,30.0,30.0
mean,1.503960e+09,2016-04-26 12:00:00,12520.6,1877.0,59.8,6.0,828.4,227.3,19.8,40.0
min,1.503960e+09,2016-04-12 00:00:00,9705.0,1728.0,27.0,4.1,539.0,130.0,5.0,16.0
25%,1.503960e+09,2016-04-19 06:00:00,10635.2,1790.2,44.5,5.1,730.5,205.2,11.2,30.2
50%,1.503960e+09,2016-04-26 12:00:00,12438.0,1848.0,58.0,5.7,790.0,219.0,19.5,38.5
75%,1.503960e+09,2016-05-03 18:00:00,13991.2,1956.5,75.5,6.4,834.5,250.8,26.2,46.0
max,1.503960e+09,2016-05-11 00:00:00,18134.0,2159.0,93.0,11.7,1218.0,328.0,46.0,78.0
std,0.000000e+00,NaN,2099.4,105.0,19.0,1.7,173.6,40.5,10.2,14.6


### Weekday vs Weekend

In [15]:
display(bi.weekday_weekend_summary(df).round(1))
bi.weekday_weekend_chart(df).show()

,metric,days_a,days_b,avg_a,avg_b
0,steps,22,8,12795.8,11763.9
1,calories,22,8,1893.3,1832.0
2,active_minutes,22,8,58.9,62.4


### Best and Worst Day

In [16]:
display(bi.best_worst_days(df))
bi.best_worst_chart(df, "steps").show()

,metric,best_date,best_value,worst_date,worst_value
0,steps,2016-04-27,18134.000000,2016-04-17,9705.000000
1,calories,2016-04-27,2159.000000,2016-04-17,1728.000000
2,active_minutes,2016-04-22,93.000000,2016-04-21,27.000000
3,sleep_hours,2016-04-17,11.666667,2016-04-26,4.083333


### Activity Levels

In [17]:
display(bi.activity_level_totals(df).round(1))
bi.activity_levels_chart(df).show()

,level,avg_minutes,share_pct
0,Sedentary,828.4,74.3
1,Lightly active,227.3,20.4
2,Fairly active,19.8,1.8
3,Very active,40.0,3.6


### Unusual Days

In [18]:
display(bi.significant_changes(df).round(1))
bi.significant_change_chart(df, "steps").show()

/var/folders/vy/dlq37vkd5sbc9j3p_q8ns0lw0000gp/T/ipykernel_54145/2354781316.py:1: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(bi.significant_changes(df).round(1))


,Date,metric,value,usual,change_pct,z_score,direction
0,2016-05-08,sleep_hours,9.9,6.0,64.9,2.3,Increase
1,2016-05-06,active_minutes,30.0,59.8,-49.8,-1.6,Decrease
2,2016-04-29,active_minutes,28.0,59.8,-53.2,-1.7,Decrease
3,2016-04-27,steps,18134.0,12520.6,44.8,2.7,Increase
4,2016-04-27,calories,2159.0,1877.0,15.0,2.7,Increase
5,2016-04-27,active_minutes,89.0,59.8,48.8,1.5,Increase
6,2016-04-22,active_minutes,93.0,59.8,55.5,1.7,Increase
7,2016-04-21,active_minutes,27.0,59.8,-54.8,-1.7,Decrease
8,2016-04-19,calories,2035.0,1877.0,8.4,1.5,Increase
9,2016-04-17,sleep_hours,11.7,6.0,94.3,3.4,Increase


### Compare Periods

In [19]:
summary = bi.compare_periods(df, ("2016-04-12", "2016-04-25"), ("2016-04-26", "2016-05-09"))
display(summary.round(1))
bi.compare_periods_chart(summary, "Weeks 1-2", "Weeks 3-4").show()

,metric,days_a,days_b,avg_a,avg_b,change_pct,trend
0,steps,14,14,11993.6,13052.3,8.8,Improved
1,calories,14,14,1856.3,1905.6,2.7,Improved
2,active_minutes,14,14,55.6,62.9,13.1,Improved


### Missing Data (this should not raise an error)

In [20]:
empty = df.iloc[0:0]      # no rows at all
short = df.head(2)        # too few days to compare

for name, d in [("empty", empty), ("2 days", short)]:
    bi.weekday_weekend_chart(d)
    bi.best_worst_chart(d, "steps")
    bi.activity_levels_chart(d)
    bi.significant_change_chart(d, "steps")
    print(name, "- no errors")

# a person whose tracker wasn't worn on their first 3 days
not_worn = FitbitData().load().clean().for_user(6117666160).head(3)
bi.best_worst_days(not_worn)[["metric", "best_value"]]    # values should be NaN, not a crash

empty - no errors
2 days - no errors


,metric,best_value
0,steps,NaN
1,calories,NaN
2,active_minutes,NaN
3,sleep_hours,NaN
